In [0]:
%pip install google-genai youtube-transcript-api google-api-python-client

In [0]:
dbutils.library.restartPython()

In [0]:
%skip
Select Channel name
Give Course_ID
Give Course Name
Need to find the Playlist ID then need to put in the code

In [0]:
from googleapiclient.discovery import build

# Setup YouTube API
YOUTUBE_API_KEY = dbutils.secrets.get(scope="courseify", key="youtube-api")
youtube = build('youtube', 'v3', developerKey=YOUTUBE_API_KEY)

# User Inputs for Playlist Strategy
COURSE_ID = "py_jenny" 
COURSE_SUBJECT = "Python"
CHANNEL_NAME = "JennyslecturesCSIT"
PLAYLIST_ID = "PLdo5W4Nhv31bZSiqiOL5ta39vSnBxpOPT" 

# Get all videos from the Playlist
def get_playlist_videos(playlist_id):
    videos = []
    next_page_token = None
    
    while True:
        request = youtube.playlistItems().list(
            part="snippet",
            playlistId=playlist_id,
            maxResults=50,
            pageToken=next_page_token
        )
        response = request.execute()
        
        for item in response['items']:
            title = item['snippet']['title']
            # Skip private/deleted videos
            if title == "Private video" or title == "Deleted video":
                continue
                
            videos.append({
                "video_id": item['snippet']['resourceId']['videoId'],
                "title": title
            })
            
        next_page_token = response.get('nextPageToken')
        if not next_page_token:
            break
            
    return videos

# Main Import Function
def import_playlist_to_db():
    print(f"Fetching videos from playlist: {PLAYLIST_ID}...")
    videos = get_playlist_videos(PLAYLIST_ID)
    print(f"Found {len(videos)} videos in the playlist.")
    
    data_to_insert = []
    
    for index, vid in enumerate(videos):
        v_id = vid['video_id']
        v_title = vid['title']
        topic_id = f"topic_{index}" # Unique topic ID for each video
        
        # Check if already exists in Database
        exists = spark.sql(f"SELECT 1 FROM courseify.default.courseify_data WHERE video_links = '{v_id}'").count()
        if exists > 0:
            print(f"⏩ Video '{v_title}' already in DB. Skipping.")
            continue
            
        print(f"✅ Adding: {v_title}")
        data_to_insert.append((
            COURSE_ID, COURSE_SUBJECT, topic_id, v_title, "", 
            v_id, CHANNEL_NAME, "", # description empty initially
            False, False, "", "" # 👈 THE FIX: Changed 'None' to empty strings '""'
        ))
        
    # Save to Delta Table
    if data_to_insert:
        columns = ["id", "title", "topic_id", "topic", "suggested_name", "video_links", "channel_name", "description", "notes", "qa", "qa_data", "notes_data"]
        df = spark.createDataFrame(data_to_insert, schema=columns)
        df.write.format("delta").mode("append").saveAsTable("courseify.default.courseify_data")
        print(f"\n🎉 Success! {len(data_to_insert)} videos successfully added to the database.")
    else:
        print("\n✅ All videos are already up to date in the database.")

# Run the importer
import_playlist_to_db()

In [0]:
%sql
select * from courseify.default.courseify_data